# RAG + LLM Evaluation Runner

This notebook mirrors `naive_llm.ipynb`, but augments each PR prompt with top-k retrieved guideline/review chunks (payload text only) using Strategy 2 query generation and Qdrant retrieval.

In [ ]:
import os, json, time
from dotenv import load_dotenv
from pathlib import Path
from groq import Groq

ROOT = Path("..").resolve()
EVAL_PATH = ROOT / 'data' / 'processed' / 'evaluation.json'
OUT_DIR = ROOT / 'output'
RAW_OUT = OUT_DIR / 'rag_llm_raw_responses.txt'
ZERO_RESPONSE_PR = OUT_DIR / 'zero_response_PRs.txt'
PARSED_OUT = OUT_DIR / 'llm_reviews.json'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = 'openai/gpt-oss-20b'
BATCH_SIZE = 1
MAX_CHARS_PER_FILE = 8000

# evaluation.json range controls (inclusive, 0-based)
STARTING = 0
ENDING = 1

# Token controls
MODEL_CONTEXT_LIMIT = 8192
MAX_OUTPUT_TOKENS = 800
INPUT_TOKEN_BUDGET = int(MODEL_CONTEXT_LIMIT * 0.75) - MAX_OUTPUT_TOKENS
CHARS_PER_TOKEN = 2.5

# Retrieval controls
RETRIEVAL_TOP_K = 5

load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY_V2']
GROQ_API_URL = os.environ.get('GROQ_API_URL', 'https://api.groq.com')
client = Groq(api_key=GROQ_API_KEY, base_url=GROQ_API_URL)

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from rag_model.query_strategy import query_strategy

In [ ]:
def load_evaluation(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return [r for r in data if isinstance(r, dict) and r.get('source_file')]

def resolve_source_path(source_file: str):
    p = Path(source_file)
    if p.is_absolute() and p.exists():
        return p
    p1 = ROOT / source_file
    if p1.exists():
        return p1
    p2 = ROOT / 'data' / 'processed' / source_file
    if p2.exists():
        return p2
    p3 = ROOT / 'data' / 'processed' / 'evaluation_files' / p.name
    return p3

def read_source_text(source_file: str):
    p = resolve_source_path(source_file)
    text = p.read_text(encoding='utf-8', errors='ignore')
    if len(text) <= MAX_CHARS_PER_FILE:
        return text
    half = MAX_CHARS_PER_FILE // 2
    return text[:half] + '\n\n...TRUNCATED...\n\n' + text[-half:]

def est_tokens(text: str):
    return max(1, int(len(text) / CHARS_PER_TOKEN))

records = load_evaluation(EVAL_PATH)
records = records[STARTING:ENDING + 1]
print('selected_records:', len(records), 'range:', STARTING, 'to', ENDING, '(inclusive)')

In [ ]:
def build_prompt(batch):
    header = """TASK: Detect STRICT PEP 8-style violations in Python code using both source code and retrieved guidance chunks.

PROCESSING RULES:
- Process EACH PR independently.
- After finishing ONE PR, immediately output its JSON object.
- Do NOT wait for other PRs.
- Do NOT analyze across PRs.
- Use RETRIEVED_PAYLOAD_CHUNKS as supporting guidance only.

VIOLATION TYPES (ONLY these):

1) naming_convention:
   - Functions, variables, parameters -> must be snake_case
   - Classes -> must be PascalCase (CapWords)
   - Constants -> must be UPPER_CASE

2) indentation:
   - Indentation must be exactly 4 spaces per level
   - Flag tabs, non-multiple-of-4 indentation, inconsistent blocks

3) unused_import:
   - Imported module/symbol not referenced anywhere in file

4) mutable_default:
   - Function parameters using [] or {} as default values

5) documentation_formatting:
   - Docstring indentation inconsistent with block level
   - Misaligned triple quotes
   - Broken or inconsistent docstring structure

CONSTRAINTS:
- Max 5 findings per PR
- Detect ONLY clear, high-confidence violations
- Do NOT guess or infer
- Be precise: exact line_number

REVIEW COMMENT STYLE:
- Short, technical, actionable
- Mention issue + direct fix
- No explanations

OUTPUT (STRICT):
Return ONLY a JSON array:
[
  {
    \"PR_ID\": \"string\",
    \"llm_reviews\": [
      {
        \"line_number\": int,
        \"violation_category\": \"string\",
        \"review_comment\": \"string\"
      }
    ]
  }
]

CRITICAL:
- No markdown
- No explanations
- No reasoning text
- Output immediately per PR
"""

    blocks = [header]
    for item in batch:
        chunks = item.get('retrieved_chunks', [])
        chunk_block = '\n\n'.join([f'[Chunk {idx+1}]\n{c}' for idx, c in enumerate(chunks)])
        if not chunk_block:
            chunk_block = '[No retrieved payload chunks available]'

        blocks.append(
f"""RETRIEVED_PAYLOAD_CHUNKS:
{chunk_block}
END

PR:
ID: {item['id']}
CODE:
{item['source_code']}
"""
        )

    return '\n'.join(blocks)

def call_groq(prompt):
    return client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )

In [ ]:
# Single-sample run (no file writes)
sample = records[0]
entry = {
    'id': sample['id'],
    'repo': sample.get('repo'),
    'source_file': sample['source_file'],
    'source_code': read_source_text(sample['source_file'])
}
result = query_strategy(entry, top_k=RETRIEVAL_TOP_K, source_root=ROOT, evaluation_files_dir=ROOT / 'data' / 'processed' / 'evaluation_files')
entry['query_text'] = result['query_text']
entry['retrieved_chunks'] = result['retrieved_chunks']
prompt = build_prompt([entry])
print('=== QUERY TEXT ===')
print(entry['query_text'])
print('\\n=== RETRIEVED CHUNK COUNT ===', len(entry['retrieved_chunks']))
print('\\n=== PROMPT (truncated 2500 chars) ===')
print(prompt[:2500])
print('\\n=== CALLING GROQ ===')
resp = call_groq(prompt)
content = resp.choices[0].message.content
print('\\n=== RESPONSE ===')
print(content)

In [ ]:
print(prompt)

In [ ]:
batch_size = BATCH_SIZE
dry_run = False

prepared = []
for r in records:
    entry = {
        'id': r['id'],
        'repo': r.get('repo'),
        'source_file': r['source_file'],
        'source_code': read_source_text(r['source_file'])
    }
    try:
        result = query_strategy(entry, top_k=RETRIEVAL_TOP_K, source_root=ROOT, evaluation_files_dir=ROOT / 'data' / 'processed' / 'evaluation_files')
        qtext = result['query_text']
        chunks = result['retrieved_chunks']
    except Exception as e:
        qtext = f'<query/retrieval error: {e}>'
        chunks = []
    entry['query_text'] = qtext
    entry['retrieved_chunks'] = chunks
    prepared.append(entry)

id_to_repo = {x['id']: x['repo'] for x in prepared}
zero_response_prs = set()

i = 0
batch_no = 0
while i < len(prepared):
    time.sleep(2)
    batch = []
    while i < len(prepared) and len(batch) < batch_size:
        batch.append(prepared[i])
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)

    if dry_run:
        content = '<dry run>'
    else:
        resp = call_groq(prompt)
        content = resp.choices[0].message.content

    print(f'\\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    pr_ids = [p['id'] for p in batch]
    print(f'PR ids: {pr_ids}')
    print(f'content_type: {type(content).__name__}')

    is_empty = False
    if isinstance(content, str):
        content_stripped = content.strip()
        print(f'content_len: {len(content)}')
        print(f'is_empty_after_strip: {len(content_stripped) == 0}')
        print('response_preview:')
        print(content[:1200])
        if len(content_stripped) == 0:
            is_empty = True
    else:
        print('response_preview_non_str:')
        print(content)
        is_empty = True

    if is_empty:
        print(f'Empty response detected for batch {batch_no}')
        zero_response_prs.update(pr_ids)

    content_to_write = content if isinstance(content, str) else str(content)

    # Keep exact header/body format used in naive_llm
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {pr_ids}\n')
        rf.write(f'repos: {[id_to_repo[p_id] for p_id in pr_ids]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f'Batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}')

if zero_response_prs:
    with open(ZERO_RESPONSE_PR, 'w', encoding='utf-8') as f:
        for pr_id in sorted(zero_response_prs):
            f.write(f'{pr_id}\n')
    print(f'Wrote {len(zero_response_prs)} zero-response PR ids to {ZERO_RESPONSE_PR}')
else:
    print('No zero-response PRs detected.')